# 00. Agent 是什么

在进入 ReAct、Tool Use、Planning 之前，先建立 Agent 最核心的心智模型。

**Agent ≠ Model。** Model 是「给定输入，返回输出」的一次性调用。Agent 在此基础上多了三个能力：

1. **环境感知** — Agent 有自己的状态，能记住之前发生了什么。
2. **行动能力** — Agent 不只是「说」，还能「做」：调工具、发请求、写文件。
3. **控制循环** — Agent 不是一次调用，而是一个 loop：观察 → 思考 → 行动 → 再观察。

> **检查点**：能用一句话描述 Model 和 Agent 的区别。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 最简 Agent 循环
from dataclasses import dataclass

@dataclass
class MinimalAgent:
    goal: str
    state: dict
    max_steps: int = 5
    
    def run(self):
        for step in range(1, self.max_steps + 1):
            print(f"Step {step}: goal={self.goal}, state={self.state}")
            # 简单的终止条件
            if self.state.get("done"):
                print("  -> Done!")
                break
            # 模拟一次行动
            self.state["counter"] = self.state.get("counter", 0) + 1
            if self.state["counter"] >= 3:
                self.state["done"] = True

agent = MinimalAgent(goal="count to 3", state={})
agent.run()

Step 1: goal=count to 3, state={}
Step 2: goal=count to 3, state={'counter': 1}
Step 3: goal=count to 3, state={'counter': 2}
Step 4: goal=count to 3, state={'counter': 3, 'done': True}
  -> Done!


## Agent 的四要素

| 要素 | 含义 | 示例 |
|------|------|------|
| **Model/Controller** | 决策核心——下一步做什么 | LLM、规则引擎、RL policy |
| **State/Memory** | 当前知道什么、记住什么 | goal、facts、history |
| **Tools/Actions** | 能做什么 | search、calculator、bash |
| **Loop** | 何时开始、何时停止 | step limit、goal achieved |

> Agent 的本质是一个带有 Context 的循环（Loop），这天生就是状态机的运行模式。目前会使用状态图（State Graph）来规范 agent loop

> **检查点**：能画出 Agent 四要素的框图，并标注它们之间的数据流。

In [3]:
# 带工具和状态的最简 Agent

def tool_search(query: str) -> str:
    """模拟搜索引擎"""
    kb = {"python": "Python created by Guido van Rossum in 1991.",
          "agent": "AI agent perceives environment and takes actions."}
    for k, v in kb.items():
        if k in query.lower():
            return v
    return "No results."

@dataclass
class ToolAgent:
    goal: str
    history: list = None
    
    def __post_init__(self):
        if self.history is None:
            self.history = []
    
    def decide(self) -> str:
        """决策：search 还是 done"""
        if len(self.history) >= 2:
            return "done"
        return "search"
    
    def run(self):
        for _ in range(5):
            action = self.decide()
            if action == "done":
                print(f"Agent done. History: {len(self.history)} items")
                break
            result = tool_search(self.goal)
            self.history.append(result)
            print(f"Search result: {result[:60]}...")

ToolAgent(goal="learn about python").run()

Search result: Python created by Guido van Rossum in 1991....
Search result: Python created by Guido van Rossum in 1991....
Agent done. History: 2 items


## 真实 API 实验

同样的两个问题，一次裸调模型，一次把工具结果喂给模型 — 直观感受「Model ≠ Agent」。


In [4]:
# Model vs Agent：工具体验差异（真实 API）
q = "DeepSeek 的母公司叫什么？它在哪个城市？"

# 裸模型：只能凭训练数据回答
raw = chat(client, q, max_tokens=150)
print("【裸模型】", raw, "\n")

# Agent 模式：先由工具（此处模拟 search）提供观察，模型基于观察回答
observation = "[search 工具返回] DeepSeek 是杭州深度求索人工智能基础技术研究有限公司（High-Flyer/幻方量化旗下），总部位于杭州。"
grounded = chat(client, f"工具观察结果：{observation}\n\n基于观察回答：{q}", max_tokens=150)
print("【Agent（工具+模型）】", grounded)


【裸模型】  



【Agent（工具+模型）】 DeepSeek 的母公司是 **杭州深度求索人工智能基础技术研究有限公司**（隶属于幻方量化/High-Flyer），总部位于 **杭州**。


## 练习

1. 修改 `MinimalAgent` 的终止条件：让它根据 goal 是否「达成」来停止，而不是计步。
2. 给 `ToolAgent` 增加一个 `tool_calculator`（安全地计算算术表达式），并修改 `decide()` 使其能选择使用哪个工具。
3. 思考：如果 Agent 的工具调用永远返回错误，它应该怎么办？这对应什么工程问题？

    - 熔断机制：对同一个工具/调用模式设置上限直到阈值
    - 换路径：换参数、工具、降级方案等
    - 归因：区分瞬时故障、权限/配置、参数错误等，以便后续能找到问题

> **检查点**：完成第 2 题后，你应该能清晰区分 Agent 的「决策逻辑」和「工具实现」——这是理解所有 Agent 框架的基础。